In [ ]:
"""
=====================================================================
 Stochastic Real-Time Task Mapping for Heterogeneous Computing
=====================================================================
Modele : temps d'execution B_lj ~ Gamma(alpha_j, n_l*beta_lj/f_j),
discretise en S scenarios. Trois formulations (Mean / Worst-Case /
CVaR) + deux heuristiques (Greedy / Modulo), validees par Monte Carlo.

Metriques  
    tau*      : makespan optimal retourne par le solveur
    MC-mean   : makespan moyen sur N tirages Monte Carlo
    Q95       : quantile 95% du makespan
    MC-worst  : pire makespan observe (argument worst-case)

>>> POUR RECONFIGURER 
=====================================================================
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp

plt.rcParams.update({
    'font.size':        16,   # taille de base de tout le texte
    'axes.titlesize':   18,   # titre du graphe
    'axes.labelsize':   16,   # labels des axes (x et y)
    'xtick.labelsize':  14,   # chiffres sur l'axe x
    'ytick.labelsize':  14,   # chiffres sur l'axe y
    'legend.fontsize':  14,   # texte de la légende
    'lines.linewidth':  2.2,  # épaisseur des courbes
    'lines.markersize': 7,    # taille des marqueurs
    'figure.dpi':       150,  # netteté
    'savefig.dpi':      200,  # netteté à l'enregistrement
    'savefig.bbox':     'tight',  # rogne les marges blanches
})
# =====================================================================
#  PARAMETRES  
# =====================================================================
SEED        = 42
L           = 8            # nombre de taches
S           = 500          # scenarios de construction
N_MC        = 10000       # tirages Monte Carlo
GAMMA_CVAR  = 0.95         # niveau de confiance CVaR (note gamma dans l'article)

# Balayage de gamma pour l'analyse de sensibilite (dans (0,1), bornes exclues).
GAMMA_SWEEP = list(np.round(np.linspace(0.05, 0.99, 20), 3))
FIG_DIR     = "figures_simulation"

PROC_TYPES  = ["CPU", "CPU", "GPU", "GPU", "FPGA"]   # -> J = len(PROC_TYPES)
FREQ_GHZ    = [3.5,   3.0,   1.5,   1.8,   0.3]      # f_j
CAP_MS      = [55.0,  50.0,  65.0,  60.0,  45.0]     # d_j

# Parametre de forme alpha par type.
#ALPHA_BY_TYPE       = {"CPU": 8.0, "GPU": 6.0, "FPGA": 30.0}
ALPHA_BY_TYPE       = {"CPU": 8.0, "GPU": 6, "FPGA": 30.0}
N_INSTR             = [12, 8, 15, 6, 10, 14, 9, 11]            
MEAN_TARGET_BY_TYPE = {"CPU": 11.0, "GPU": 8.0, "FPGA": 6.0}   # E[B] cible
AFFINITY_RANGE      = (0.85, 1.15)

# Solveur MILP 
SOLVER_PREFS = ["GUROBI", "MOSEK", "CBC", "SCIP", "GLPK_MI"]

# Couleurs coherentes entre toutes les figures
COLORS = {"Mean":"green", "WC":"red", "CVaR":"purple",
          "Modulo":"skyblue", "Greedy":"orange"}
# =====================================================================


def pick_solver():
    return cp.MOSEK

# ---------------------------------------------------------------------
#  1. Donnees synthetiques 
# ---------------------------------------------------------------------
def build_data(seed=SEED):
    rng = np.random.default_rng(seed)
    J = len(PROC_TYPES)
    assert len(FREQ_GHZ) == J == len(CAP_MS)
    assert len(N_INSTR) == L

    f     = np.array(FREQ_GHZ, float)
    d     = np.array(CAP_MS,   float)
    n     = np.array(N_INSTR,  float)
    alpha = np.array([ALPHA_BY_TYPE[t] for t in PROC_TYPES], float)

    affinity = rng.uniform(*AFFINITY_RANGE, size=(L, J))
    E_target = np.array([[MEAN_TARGET_BY_TYPE[PROC_TYPES[j]] * affinity[l, j]
                          for j in range(J)] for l in range(L)])
    beta   = E_target * f[None, :] / (alpha[None, :] * n[:, None])
    scaleB = n[:, None] * beta / f[None, :]
    E_B    = alpha[None, :] * scaleB          # (L,J) E[B_lj]

    # Scenarios de construction
    B = np.empty((L, J, S))
    for l in range(L):
        for j in range(J):
            B[l, j, :] = rng.gamma(alpha[j], scaleB[l, j], size=S)
    p = np.full(S, 1.0 / S)

    # Monte Carlo (independant)
    MC = np.empty((L, J, N_MC))
    for l in range(L):
        for j in range(J):
            MC[l, j, :] = rng.gamma(alpha[j], scaleB[l, j], size=N_MC)

    return dict(J=J, f=f, d=d, n=n, alpha=alpha, scaleB=scaleB,
                E_B=E_B, B=B, p=p, MC=MC)


def afficher_donnees(data):
    J, E_B, d = data["J"], data["E_B"], data["d"]
    print("="*56)
    print(f"CONFIG : L={L} taches, J={J} proc, S={S} scenarios, N_MC={N_MC}")
    print("="*56)
    df = pd.DataFrame(np.round(E_B, 2),
                      index=[f"T{l+1}" for l in range(L)],
                      columns=[f"{PROC_TYPES[j]}{j+1}" for j in range(J)])
    print("\nE[B_lj] (ms) - duree moyenne de chaque tache par processeur :")
    print(df.to_string())
    print("\nCapacites d_j (ms) :", dict(
        zip([f"{PROC_TYPES[j]}{j+1}" for j in range(J)], d)))
    print("="*56 + "\n")


# ---------------------------------------------------------------------
#  2. Formulations d'optimisation (CVXPY / MILP)
# ---------------------------------------------------------------------
def _round(v):
    return (np.array(v) > 0.5).astype(float)


def solve_mean(data, solver):
    J, B, p, d = data["J"], data["B"], data["p"], data["d"]
    Eload = (B * p[None, None, :]).sum(axis=2)         # (L,J)
    x   = cp.Variable((L, J), boolean=True)
    tau = cp.Variable(nonneg=True)
    cons = [cp.sum(x, axis=1) == 1]
    for j in range(J):
        load = cp.sum(cp.multiply(x[:, j], Eload[:, j]))
        cons += [load <= tau, load <= d[j]]
    cp.Problem(cp.Minimize(tau), cons).solve(solver=solver)
    return _round(x.value), float(tau.value)


def solve_worstcase(data, solver):
    J, B, d = data["J"], data["B"], data["d"]
    x   = cp.Variable((L, J), boolean=True)
    tau = cp.Variable(nonneg=True)
    cons = [cp.sum(x, axis=1) == 1]
    for j in range(J):
        for s in range(S):
            load = cp.sum(cp.multiply(x[:, j], B[:, j, s]))
            cons += [load <= tau, load <= d[j]]
    cp.Problem(cp.Minimize(tau), cons).solve(solver=solver)
    return _round(x.value), float(tau.value)


def solve_cvar(data, solver, gamma_cvar=GAMMA_CVAR):
    J, B, p, d = data["J"], data["B"], data["p"], data["d"]
    x    = cp.Variable((L, J), boolean=True)
    eta  = cp.Variable()
    zeta = cp.Variable(S, nonneg=True)
    taus = cp.Variable(S, nonneg=True)
    cons = [cp.sum(x, axis=1) == 1]
    for s in range(S):
        for j in range(J):
            load = cp.sum(cp.multiply(x[:, j], B[:, j, s]))
            cons += [load <= taus[s], load <= d[j]]
        cons += [zeta[s] >= taus[s] - eta]
    obj = eta + (1.0/(1.0-gamma_cvar)) * cp.sum(cp.multiply(p, zeta))
    cp.Problem(cp.Minimize(obj), cons).solve(solver=solver)
    return _round(x.value), float(obj.value)


# ---------------------------------------------------------------------
#  3. Heuristiques
# ---------------------------------------------------------------------
def solve_greedy(data):
    J, E_B = data["J"], data["E_B"]
    x = np.zeros((L, J)); cur = np.zeros(J)
    # taches par duree esperee decroissante (comme ton ancien code)
    order = np.argsort(-E_B.min(axis=1))
    for l in order:
        j = int(np.argmin(cur + E_B[l, :])); x[l, j] = 1; cur[j] += E_B[l, j]
    return x, None


def solve_modulo(data):
    J = data["J"]
    x = np.zeros((L, J))
    for l in range(L):
        x[l, l % J] = 1
    return x, None


# ---------------------------------------------------------------------
#  4. Monte Carlo
# ---------------------------------------------------------------------
def monte_carlo(x, data):
    MC = data["MC"]
    load = np.einsum('lj,ljn->jn', x, MC)      # (J,N)
    mk   = load.max(axis=0)                     # (N,)
    return dict(makespan=mk, mc_mean=mk.mean(),
                q95=np.quantile(mk, 0.95), mc_worst=mk.max())


def charges_esperees(x, data):
    """Charge esperee par processeur pour la figure des charges/Gantt."""
    return (x * data["E_B"]).sum(axis=0)       # (J,)


# ---------------------------------------------------------------------
#  5. Figures 
# ---------------------------------------------------------------------
def fig_charges(charges, taus, data):
    J = data["J"]; xpos = np.arange(J); w = 0.15
    order = ["Mean","WC","CVaR","Modulo","Greedy"]
    plt.figure(figsize=(12, 6))
    for k, name in enumerate(order):
        c = charges[name]; mx = c.max()
        plt.bar(xpos + (k-2)*w, c, w, color=COLORS[name],
                label=f"{name} (max={mx:.1f})")
        plt.axhline(mx, color=COLORS[name], ls='--', alpha=0.6)
    plt.xticks(xpos, [f"{PROC_TYPES[j]}{j+1}" for j in range(J)])
    plt.ylabel("Mean execution time (ms)"); plt.xlabel("Processors")
   # plt.title("Processor workloads and maximum execution time")
    plt.legend(); plt.grid(axis='y', ls='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "charges_par_processeur.png"), dpi=150)
    plt.close()


#def fig_gantt(x, data, title, fname):
   # J = data["J"]; E_B = data["E_B"]
    #jof = np.argmax(x, axis=1)
    #fig, ax = plt.subplots(figsize=(10, 5))
    #t = np.zeros(J)
    #for l in range(L):
    #    j = jof[l]; dur = E_B[l, j]
    #    ax.barh(j, dur, left=t[j], edgecolor='black')
     #   ax.text(t[j]+dur/2, j, f"T{l+1}", ha='center', va='center', fontsize=8)
     #   t[j] += dur
    #mk = t.max()
    #ax.axvline(mk, color='red', ls='--', label=f"tau*={mk:.1f}")
    #ax.set_yticks(range(J)); ax.set_yticklabels([f"{PROC_TYPES[j]}{j+1}" for j in range(J)])
    #ax.set_xlabel("Mean execution time (ms)"); ax.set_ylabel("Processors")
    #ax.set_title(title); 
    #ax.legend(); 
    #ax.grid(True, alpha=0.3)
    #plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, fname), dpi=150); plt.close()


def fig_hist_mc(mks, taus, etas):
    plt.figure(figsize=(12, 6))
    ymax = 0
    for name, mk in mks.items():
        counts, _, _ = plt.hist(mk, bins=60, alpha=0.45,
                                color=COLORS[name], label=name)
        ymax = max(ymax, counts.max())
    # Maximum observe (MC-worst) de chaque methode : trait plein a la
    # limite de sa queue -> lecture directe du pire-cas empirique.
    for name, mk in mks.items():
        w = mk.max()
        plt.axvline(w, color=COLORS[name], ls='-', lw=1.6, alpha=0.9)
    # tau* optimaux (traits pointilles, references theoriques)
    plt.axvline(taus["Mean"], color='green',  ls='--', lw=1, alpha=0.7)
    plt.axvline(taus["WC"],   color='red',    ls='--', lw=1, alpha=0.7)
    plt.axvline(etas,         color='purple', ls='--', lw=1, alpha=0.7)
    plt.xlabel("Maximum execution time (ms)"); plt.ylabel("Frequency")
    #plt.title(f"Monte Carlo distribution of makespan ({N_MC} draws)\n"
             # "solid lines = per-method observed maximum (MC-worst)")
    plt.legend(loc='upper right'); plt.grid(True, alpha=0.3)
    plt.ylim(0, ymax*1.05)
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "histogrammes_mc.png"), dpi=150); plt.close()


#def fig_conv_max(mks, batch=50):
    #plt.figure(figsize=(12, 6))
    #for name, mk in mks.items():
        #cummax = np.maximum.accumulate(mk)
        #idx = np.arange(batch, len(mk)+1, batch)
        #plt.plot(idx, cummax[idx-1], color=COLORS[name], label=name)
    #plt.xlabel("Number of Monte Carlo simulations")
    #plt.ylabel("Observed maximum execution time (ms)")
   # plt.title("Evolution of the maximum execution time (cumulative max)")
    #plt.legend(); plt.grid(True, alpha=0.3)
    #plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "convergence_max.png"), dpi=150); plt.close()


#def fig_conv_mean(mks, batch=50):
   # plt.figure(figsize=(12, 6))
    #for name, mk in mks.items():
       # cummean = np.cumsum(mk)/np.arange(1, len(mk)+1)
       # idx = np.arange(batch, len(mk)+1, batch)
       # plt.plot(idx, cummean[idx-1], color=COLORS[name], label=name)
    #plt.xlabel("Number of Monte Carlo simulations")
    #plt.ylabel(r"Cumulative mean of makespan  $\mathbb{E}[\tau_{max}]$")
    #plt.title("Monte Carlo convergence of the mean makespan")
    #plt.legend(); plt.grid(True, alpha=0.3)
    #plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "convergence_mean.png"), dpi=150); plt.close()


#def fig_cvar_sensitivity(data, solver, levels):
   # rows = []
    #for g in levels:
       # x, _ = solve_cvar(data, solver, g)
      #  mc = monte_carlo(x, data)
      #  rows.append((g, mc["mc_mean"], mc["q95"]))
    #r = np.array(rows)
    #fig, ax = plt.subplots(figsize=(10, 5))
    #ax.plot(r[:,0], r[:,1], 'o-', color="#E67E22", label="MC-mean")
    #ax.plot(r[:,0], r[:,2], 's-', color="#8E44AD", label="Q95")
    #ax.set_xlabel(r"CVaR confidence level $\gamma$")
    #ax.set_ylabel("Makespan (ms)")
    #ax.legend(frameon=False)
   # ax.set_title(r"CVaR sensitivity to confidence level $\gamma$")
    #ax.spines[['top', 'right']].set_visible(False)
    #plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "cvar_sensitivity.png"), dpi=150); plt.close()
    #return r


# ---------------------------------------------------------------------
#  6. Pipeline principal
# ---------------------------------------------------------------------
def main():
    os.makedirs(FIG_DIR, exist_ok=True)
    solver = pick_solver()
    print(f"Solveur MILP : {solver}\n")

    data = build_data()
    afficher_donnees(data)

    # Resolution des 5 methodes 
    x, taus, charges, mks, metrics = {}, {}, {}, {}, {}
    x["Mean"], taus["Mean"] = solve_mean(data, solver)
    x["WC"],   taus["WC"]   = solve_worstcase(data, solver)
    x["CVaR"], taus["CVaR"] = solve_cvar(data, solver, GAMMA_CVAR)
    x["Greedy"], _ = solve_greedy(data)
    x["Modulo"], _ = solve_modulo(data)
    for name in ["Greedy", "Modulo"]:
        taus[name] = None

    for name in x:
        charges[name] = charges_esperees(x[name], data)
        mc = monte_carlo(x[name], data)
        mks[name] = mc["makespan"]
        metrics[name] = mc

    # Tableau comparatif  
    table = pd.DataFrame({
        "Method": ["Mean","WC","CVaR","Greedy","Modulo"],
        "tau*":   [taus["Mean"], taus["WC"], taus["CVaR"], np.nan, np.nan],
        "MC-mean":[metrics[m]["mc_mean"]  for m in ["Mean","WC","CVaR","Greedy","Modulo"]],
        "Q95":    [metrics[m]["q95"]      for m in ["Mean","WC","CVaR","Greedy","Modulo"]],
        "MC-worst":[metrics[m]["mc_worst"] for m in ["Mean","WC","CVaR","Greedy","Modulo"]],
    })
    pd.set_option("display.float_format", lambda v: f"{v:.2f}")
    print("===== TABLEAU COMPARATIF =====")
    print(table.to_string(index=False))
    table.to_csv(os.path.join(FIG_DIR, "results.csv"), index=False)

    # Affichage explicite (evite toute troncature de la sortie) :
    print("\n--- Detail par methode (garanti non tronque) ---")
    for m in ["Mean","WC","CVaR","Greedy","Modulo"]:
        tau_s = f"{taus[m]:.2f}" if taus[m] is not None else "  --  "
        print(f"  {m:7s} | tau*={tau_s} | MC-mean={metrics[m]['mc_mean']:.2f} "
              f"| Q95={metrics[m]['q95']:.2f} | MC-worst={metrics[m]['mc_worst']:.2f}")

    # Argument worst-case reformule 
    wc_worst = metrics["WC"]["mc_worst"]
    best = min(metrics[m]["mc_worst"] for m in metrics)
    print(f"\n[Worst-case] MC-worst du WC = {wc_worst:.2f} ms ; "
          f"plus bas pire-cas de toutes les methodes ? "
          f"{'OUI' if abs(wc_worst-best)<1e-9 else 'NON'}")
    print(f"[Worst-case] tau* du WC = {taus['WC']:.2f} ms (borne sur les S "
          f"scenarios de construction ; le MC-worst sur {N_MC} tirages "
          f"independants peut le depasser, ce qui est attendu).")

    #  Figures 
    fig_charges(charges, taus, data)
    #fig_gantt(x["Mean"],   data, "Gantt - Mean model",  "gantt_mean.png")
    #fig_gantt(x["WC"],     data, "Gantt - Worst-Case",  "gantt_wc.png")
    #fig_gantt(x["CVaR"],   data, "Gantt - CVaR",        "gantt_cvar.png")
    #fig_gantt(x["Greedy"], data, "Gantt - Greedy",      "gantt_greedy.png")
    #fig_gantt(x["Modulo"], data, "Gantt - Modulo",      "gantt_modulo.png")
    fig_hist_mc(mks, taus, taus["CVaR"])
    #fig_conv_max(mks)
    #fig_conv_mean(mks)
    #fig_cvar_sensitivity(data, solver, GAMMA_SWEEP)

    print(f"\nFigures + results.csv dans ./{FIG_DIR}/")
    return table


if __name__ == "__main__":
    main() 



    
